# The patent application abstract table (TLS203_APPLN_ABSTR)

Welcome to the TLS203_APPLN_ABSTR table in PATSTAT. This table contains the English-language abstract, if available. If there is no abstract in English, it contains the most recent abstract in another language.

As always, we start creating the PATSTAT client and accessing ORM. Then we import the TLS203_APPLN_ABSTR table.

In [1]:
from epo.tipdata.patstat import PatstatClient

# Initialize the PATSTAT client
patstat = PatstatClient(env='PROD')

# Access ORM
db = patstat.orm()

# Importing the as models
from epo.tipdata.patstat.database.models import TLS203_APPLN_ABSTR

## APPLN_ID (Primary Key)

Again, we find the unique identifier for each patent application as primary key of the table. Let's join this table with table TLS201.

In [2]:
# Import table TLS201
from epo.tipdata.patstat.database.models import TLS201_APPLN

appln_id = db.query(
    TLS203_APPLN_ABSTR.appln_id,
    TLS201_APPLN.appln_nr
).join(
    TLS201_APPLN, TLS203_APPLN_ABSTR.appln_id == TLS201_APPLN.appln_id  # Join the two table via the common appln_id attribute
).limit(20000)

appln_id_df = patstat.df(appln_id)
appln_id_df

,appln_id,appln_nr
0,532784898,202010155869
1,515847687,201821246889
2,34558523,21782885
3,34916758,23884594
4,18358641,200800066
...,...,...
19995,509841866,201910024274
19996,267500544,200820170172
19997,54310549,95391107
19998,480897775,201621386021


## APPLN_ABSTRACT_LG

Language of the abstract of the application selected for and loaded in PATSTAT.

Let's check how many applications have an abstract that is not in English. We import `func` from `sqlalchemy` to perform the `count` on the application IDs. We filter by excluding the applications that have the `appln_abstract_lg` different from 'en'.

In [3]:
from sqlalchemy import func

lg = db.query(
    TLS203_APPLN_ABSTR.appln_abstract_lg,
    func.count(TLS203_APPLN_ABSTR.appln_id).label('total_applications')
).filter(
    TLS203_APPLN_ABSTR.appln_abstract_lg != 'en'
).group_by(
    TLS203_APPLN_ABSTR.appln_abstract_lg
).order_by(
    func.count(TLS203_APPLN_ABSTR.appln_id).desc()
)

lg_df = patstat.df(lg)
lg_df

,appln_abstract_lg,total_applications
0,ko,1685398
1,de,1294816
2,zh,1158663
3,ja,996088
4,es,939827
5,fr,692191
6,pt,558149
7,ru,257785
8,tr,97058
9,uk,69700


We can check which application authorities have more distinct languages used for the abstracts of the applications filed therein.

In [4]:
lg_auth = db.query(
    TLS201_APPLN.appln_auth,
    func.count(TLS203_APPLN_ABSTR.appln_abstract_lg.distinct()).label('num_of_languages')  # Count the distinct number of title languages
).join(
    TLS201_APPLN, TLS203_APPLN_ABSTR.appln_id == TLS201_APPLN.appln_id
).group_by(
    TLS201_APPLN.appln_auth
).order_by(
    func.count(TLS203_APPLN_ABSTR.appln_abstract_lg.distinct()).desc()  
)

lg_auth_df = patstat.df(lg_auth)
lg_auth_df

,appln_auth,num_of_languages
0,CH,4
1,KR,4
2,BE,4
3,ME,4
4,WO,3
...,...,...
78,GE,1
79,CY,1
80,EG,1
81,AR,1


Out of curiosity, let's check which title languages are present among the applications filed at the EPO and the WIPO. As in table TLS202, we need to import `distinct`.

In [5]:
# Import distinct from sqlalchemy
from sqlalchemy import distinct

lg_wo_ep_ch = db.query(
    TLS201_APPLN.appln_auth,
    TLS203_APPLN_ABSTR.appln_abstract_lg.label('distinct_languages')
).distinct(  # Consider distinct appln_auth-appln_title_lg rows combinations only
).join(
    TLS201_APPLN, TLS203_APPLN_ABSTR.appln_id == TLS201_APPLN.appln_id
).filter(
    (TLS201_APPLN.appln_auth == 'WO') | (TLS201_APPLN.appln_auth == 'EP')
).order_by(
    TLS201_APPLN.appln_auth
)

lg_wo_ep_ch_df = patstat.df(lg_wo_ep_ch)
lg_wo_ep_ch_df

,appln_auth,distinct_languages
0,EP,fr
1,EP,en
2,EP,de
3,WO,en
4,WO,de
5,WO,fr


## APPLN_ABSTRACT

This is the abstract of the application. Multiple abstracts may be published for any application but only one abstract will be stored in PATSTAT, according to the following rules (first applicable rule is applied):
1. most recent (according to publication date) abstract in English 
2. most recent abstract in language of publication
3. most recent abstract in any other language.

We can show the abstracts of the applications with `appln_abstract_lg` equal to 'en'.

In [9]:
abstract = db.query(
    TLS203_APPLN_ABSTR.appln_id,
    TLS203_APPLN_ABSTR.appln_abstract_lg,
    TLS203_APPLN_ABSTR.appln_abstract
).filter(
    TLS203_APPLN_ABSTR.appln_abstract_lg == 'en'
).limit(1000)

abstract_df = patstat.df(abstract)
abstract_df

,appln_id,appln_abstract_lg,appln_abstract
0,13533288,en,An optical sensor comprises a semiconductor ch...
1,493083117,en,The invention discloses a containerized transp...
2,445922449,en,A computer-implemented method of allocating to...
3,27644605,en,PURPOSE:To provide excellent charge/discharge ...
4,588664391,en,The invention relates to a stator (100) usable...
...,...,...,...
995,612696949,en,The present invention provides a triarylamine ...
996,561975615,en,The utility model discloses a clamping device ...
997,547770126,en,The utility model relates to the technical fie...
998,8016630,en,The utility model relates to a carrier structu...
